# ClimatePulse: full pipeline on Google Colab

Runs the whole workflow from scratch: data download, all model training, evaluation, saved snapshot
and (optionally) the paper and patent PDFs. It follows the Makefile step for step; see
`README.md` ("Reproducing the results") for the same sequence on a workstation.

**Before you start**
- *Runtime → Change runtime type → GPU* speeds up training a lot (the weather model is about 2 h on CPU).
- Add your OpenAQ key as a Colab secret named `OPENAQ_API_KEY` (key icon in the left sidebar, "Notebook access" on).
  Free key: https://explore.openaq.org/register
- A full run (`QUICK = False`) takes several hours. Colab sessions can disconnect, so keep `USE_DRIVE = True`:
  downloaded data and results are then copied to Google Drive and restored on the next session.
- `QUICK = True` runs every step for 1–2 epochs and one seed, to check that the notebook works. Its numbers are **not** results.

**Reproducibility.** Seeds are fixed, but GPU/CPU type and thread counts can change results slightly, and
freshly downloaded data (OpenAQ, NOAA, GISTEMP) can differ from the data behind the paper. Numbers written
in the paper and patent text are typed in by hand and are not updated by this notebook.

## 1. Settings

In [ ]:
#@title Run settings
REPO_URL = "https://github.com/archanpaul/climate_pulse.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
QUICK = True  #@param {type:"boolean"}
SEEDS = "0 1 2 3 4"  #@param {type:"string"}
RUN_NWM_ABLATION = True  #@param {type:"boolean"}
BUILD_PDFS = False  #@param {type:"boolean"}
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/climate_pulse_runs"  #@param {type:"string"}

## 2. Get the code and install dependencies

In [ ]:
import os, pathlib, shutil, subprocess, sys

# Clone (or reuse the current directory when run from inside the repo)
if pathlib.Path("Makefile").exists() and pathlib.Path("src/config.py").exists():
    REPO = pathlib.Path.cwd()
else:
    REPO = pathlib.Path("/content/climate_pulse")
    if not REPO.exists():
        url = REPO_URL
        try:  # private repos: add a GitHub token as Colab secret GITHUB_TOKEN
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
            url = url.replace("https://", f"https://{token}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "--branch", BRANCH, url, str(REPO)], check=True)
os.chdir(REPO)
print("Repo:", REPO)
!git log --oneline -1

In [ ]:
# Install the package and test tools (Colab's preinstalled PyTorch is reused)
!pip install -q -e ".[dev]"

In [ ]:
# PyG's optional accelerator; skipped if no wheel exists for this torch/CUDA combination
import torch
tv = torch.__version__.split("+")[0]
tag = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
!pip install -q pyg-lib -f https://data.pyg.org/whl/torch-{tv}+{tag}.html || echo "pyg-lib not available for this torch build; continuing without it"

In [ ]:
# The Makefile calls .venv/bin/python; point it at this notebook's interpreter instead of creating a venv
os.makedirs(".venv/bin", exist_ok=True)
shim = pathlib.Path(".venv/bin/python")
if shim.exists() or shim.is_symlink():
    shim.unlink()
shim.write_text(f'#!/bin/sh
exec "{sys.executable}" "$@"
')
shim.chmod(0o755)
pathlib.Path(".venv/.created").touch()

# Run settings as environment variables (inherited by every `!make` command below)
if QUICK:
    os.environ["MODEL_MAX_EPOCHS"] = "2"
    SEEDS = "0"
    print("QUICK mode: 2 epochs, 1 seed. Results are for checking the notebook only.")
else:
    os.environ.pop("MODEL_MAX_EPOCHS", None)
print("Seeds:", SEEDS)

## 3. Credentials and (optional) Google Drive

In [ ]:
# OpenAQ key from Colab secrets -> .env (gitignored)
try:
    from google.colab import userdata
    key = userdata.get("OPENAQ_API_KEY")
    pathlib.Path(".env").write_text(f"DATA_OPENAQ_API_KEY={key}\n")
    print(".env written")
except Exception as e:
    print("No OPENAQ_API_KEY secret found:", e)
    print("Cached data from Drive (next cell) can be used instead; downloading OpenAQ data needs the key.")

In [ ]:
DRIVE = pathlib.Path(DRIVE_DIR)
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE.mkdir(parents=True, exist_ok=True)
    cached = DRIVE / "datasets"
    if cached.exists():
        shutil.copytree(cached, "datasets", dirs_exist_ok=True)
        print("Restored datasets from", cached)
    else:
        print("No cached datasets on Drive yet; they will be downloaded and saved after step 4.")

## 4. Data

`make download-data` reuses any cached files (including those restored from Drive); `make clean-downloaded-datasets`
forces a fresh download. The ERA5 subset for the weather model is large.

In [ ]:
!make download-data

In [ ]:
!make download-nwm-data

In [ ]:
if USE_DRIVE:
    shutil.copytree("datasets", DRIVE / "datasets", dirs_exist_ok=True)
    print("Datasets saved to", DRIVE / "datasets")

In [ ]:
!make test

## 5. El Niño

Five seeds of the TFT (with and without the calendar-year covariate) and the LSTM. The TFT variant is
chosen on mean **validation** loss; the next cell applies that choice to `configs/runtime_conf.yaml`
before the main run.

In [ ]:
!PYTHONPATH=src .venv/bin/python src/scripts/run_elnino_seeds.py --seeds {SEEDS}

In [ ]:
import json, re
sel = json.load(open("runtime/eval_artifacts/run_results/elnino_seed_selection.json"))
print("Selected TFT variant:", sel["selected_variant"], "| mean validation loss:", sel["mean_val_loss"])
conf = pathlib.Path("configs/runtime_conf.yaml")
text = re.sub(r"(?m)^(\s*tft_use_year:).*$", rf"\1 {str(sel['tft_use_year']).lower()}", conf.read_text())
conf.write_text(text)
!grep -n tft_use_year configs/runtime_conf.yaml

In [ ]:
!make train-elnino

## 6. Pollution and climate models

In [ ]:
!make train-pollution

In [ ]:
!make train-climate

## 7. Neural weather model

The main model trains first, then the TFT lead-1 forecasts are exported, then the three comparison versions
(forecast-conditioned, unconditioned, unweighted loss) train in parallel and are scored against it.

In [ ]:
!make train-nwm

In [ ]:
!make export-tft-forecasts

In [ ]:
if RUN_NWM_ABLATION:
    !make nwm-ablation
else:
    print("Skipped (RUN_NWM_ABLATION = False); the ablation table will show n/a.")

## 8. Evaluation artifacts and saved snapshot

In [ ]:
!make gen-eval-artifacts

In [ ]:
!make save-run
!make sync-paper-artifacts

In [ ]:
from IPython.display import Markdown, display
display(Markdown(pathlib.Path("runtime/eval_artifacts/metrics_summary.md").read_text()))

In [ ]:
from IPython.display import Image, display
for name in ["elnino_forecast_vs_actual", "elnino_crps_leadtime", "elnino_calibration_diagram",
             "pollution_confusion_matrix", "nwm_spatial_error_map", "climate_changepoint_regression"]:
    display(Image(f"runtime/eval_artifacts/{name}.png", width=520))

## 9. Paper and patent PDFs (optional)

Needs pandoc and XeLaTeX; installing them takes a few minutes. The text of `research_paper.md` and
`patent_application.md` is not changed by this notebook, so update the numbers in it by hand first.

In [ ]:
if BUILD_PDFS:
    !apt-get -qq install -y pandoc texlive-xetex texlive-latex-extra texlive-fonts-recommended texlive-science lmodern > /dev/null
    !make gen-research-paper-pdf
    !make gen-patent-application-pdf
else:
    print("Skipped (BUILD_PDFS = False).")

## 10. Save the results

In [ ]:
if USE_DRIVE:
    for src in ["runtime/saved", "runtime/eval_artifacts", "docs/patent/artifacts"]:
        shutil.copytree(src, DRIVE / src, dirs_exist_ok=True)
    for pdf in pathlib.Path("docs").rglob("*.pdf"):
        dst = DRIVE / "docs" / pdf.relative_to("docs")
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(pdf, dst)
    print("Results copied to", DRIVE)

# Also offer a download of the evaluation artifacts and the newest snapshot
archive = shutil.make_archive("/content/climate_pulse_results", "zip", root_dir=".", base_dir="runtime/eval_artifacts")
try:
    from google.colab import files
    files.download(archive)
except Exception:
    print("Archive:", archive)